# Logistic Regression — Annotated Lecture Notebook

**Lecture problem:** Can we predict whether a student will pass using weekly study hours, attendance, and previous score?

### Learning path

`features → linear score → sigmoid probability → threshold → class`

Training adds:

`prediction → loss → gradients → parameter update → repeat`

> **Source note:** These annotations are grounded in the supplied notebook and the lecture-note generation rules. They explain the existing code rather than replacing it with a different implementation.


## 1. Problem formulation

**Inputs:** `study_hours`, `attendance`, `previous_score`

**Target:** `pass/fail`, encoded as `1 = pass`, `0 = not pass`

**Model output:** probability of passing

**Decision:** compare the probability with a selected threshold to obtain the final class.

**Why this matters:** Before coding logistic regression, identify exactly what the model receives and what it must predict.


## Can we predict whether a student will pass using weekly study hours, attendance, and previous score?

Inputs : study_hours, attendance, previous_score
Target : pass or fail where 1 = pass and 0 = not pass 
Model Output : Probability of passing
Decesion: classify as pass when the selected threshold is reached
Goal : understand each and every step before finalizing the library implementation 

### Environment setup

This cell installs NumPy. It is environment setup rather than a logistic-regression concept, so it is not part of the conceptual pipeline.

In [45]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


## 2. NumPy import

NumPy supplies the array and numerical operations used throughout the implementation. This is supporting setup, not a new ML concept.

In [46]:
# imports 
import numpy as np

## 3. Sigmoid function

**Concept link:** Logistic regression probability transformation.

**What:** Converts the linear score into a value between 0 and 1.

**Why:** The raw linear score can be any real number; the sigmoid produces the probability-like output used for binary classification.

**Key line:** `np.clip(values, -500, 500)` limits the exponent input to avoid numerical overflow in `exp`.

The mathematical form is:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

**Common beginner mistake:** treating the linear score itself as the probability.


**Professor's Code — concept linked to the annotation above**

In [47]:
# sigmoid function 
def sigmoid(values):
    values = np.clip(values,-500,500)
    return 1 / (1 + np.exp(-values))

## 4. Binary cross-entropy loss

**Concept link:** Loss used to measure how well predicted probabilities match binary labels.

The lecture notebook describes lower binary cross-entropy as indicating better alignment between predictions and true labels.

The next cell implements that loss.

Binary Entropy Loss : It quantifies the difference between the actual class labels (0 or 1) and the predicted probabilities output by the model. The lower the binary cross-entropy value, the better the model’s predictions align with the true labels 

## 5. Binary cross-entropy implementation

**What:** Calculates the average binary cross-entropy from true labels and predicted probabilities.

**Why:** Training needs a numerical loss value to monitor how well the current parameters fit the labels.

**Important notebook detail:** The supplied line `np.clip(probabilities,1e-12, 1e-12)` clips both bounds to the same value. This is worth checking before relying on it in a production implementation; normally clipping uses a small lower bound and a value just below 1 as the upper bound. The annotation does not silently rewrite the professor's cell.


**Professor's Code — concept linked to the annotation above**

In [48]:
# calculate the binary cross entropy loss function 
def binary_cross_entropy(labels, probabilities):
    probabilities = np.clip(probabilities,1e-12, 1e-12)
    return -np.mean(labels * np.log(probabilities) + (1 - labels) * np.log(1 - probabilities))

## 6. First/manual draft of the logistic-regression class

**Concept link:** End-to-end training mechanics.

This commented-out block shows the intended training loop: compute scores, convert to probabilities, calculate error, compute gradients, update weights/bias, and record loss.

It is useful as a learning scaffold because the individual steps are visible. The next cell is the active corrected implementation.

**Professor's Code — concept linked to the annotation above**

In [49]:
# implement the logistic regression model 
# class LogisticRegressionModel:
#     def __init__(self,learning_rate=0.1,epocs=2000):
#         self.learning_rate = learning_rate
#         self.epocs= epocs
#         self.weights= None
#         self.bias = 0.0
#         self.loss_history= []

#     def fit(self,features, labels):
#         row_count, feature_count = features.shape
#         self.weights = np.zeros(features)

#         for _ in range(self.epocs):
#             scores = features @ self.weights + self.bias
#             probabilities = sigmoid(scores)
#             error = probabilities - labels
#             weight_gradient = features.T @ error / row_count
#             bias_gradient = np.mean(error)
#             self.weights -= self.learning_rate * weight_gradient # weights - self.learning_rate * weight_gradient
#             self.bias -= self.learning_rate * bias_gradient
#             self.loss_history(
#                 binary_cross_entropy(labels,probabilities)
#             )
#         return self


#     def predict_proba(self,features):
#         return sigmoid(features @ self.weights + self.bias)

#     def predict(self,features,threshold=0.5):
#         return (self.predict_proba >= threshold).astype(int)

        

## 7. Active logistic-regression implementation

**Concept link:** Training with gradient descent.

### What
The class implements three core operations:

1. `fit(...)` — learn weights and bias from training data.
2. `predict_proba(...)` — return predicted probabilities.
3. `predict(...)` — convert probabilities to class labels using a threshold.

### Training loop

`features @ weights + bias` → linear score

`sigmoid(scores)` → probabilities

`probabilities - labels` → prediction error

`features.T @ error / row_count` → weight gradient

`mean(error)` → bias gradient

`weights -= learning_rate * gradient` and `bias -= learning_rate * gradient` → parameter updates

### Why the visible fixes matter

- `np.zeros(feature_count)` creates one weight per feature.
- `.append(...)` adds each loss value to the loss-history list.
- `self.predict_proba(features)` actually calls the method before comparing the result with the threshold.

**Common beginner mistake:** confusing the model's probability output with the final thresholded class.

**Professor's Code — corrected implementation shown in the supplied notebook — concept linked to the annotation above**

In [50]:
class LogisticRegressionModel:
    def __init__(self, learning_rate=0.1, epocs=2000):
        self.learning_rate = learning_rate
        self.epocs = epocs
        self.weights = None
        self.bias = 0.0
        self.loss_history = []

    def fit(self, features, labels):
        row_count, feature_count = features.shape
        # ✅ FIX 1: Initialize weights as a 1D array matching the number of features
        self.weights = np.zeros(feature_count) 

        for _ in range(self.epocs):
            scores = features @ self.weights + self.bias
            probabilities = sigmoid(scores)
            error = probabilities - labels
            weight_gradient = features.T @ error / row_count
            bias_gradient = np.mean(error)
            
            self.weights -= self.learning_rate * weight_gradient 
            self.bias -= self.learning_rate * bias_gradient
            
            # ✅ FIX 2: Use .append() instead of trying to call the list like a function
            self.loss_history.append(
                binary_cross_entropy(labels, probabilities)
            )
        return self

    def predict_proba(self, features):
        return sigmoid(features @ self.weights + self.bias)

    def predict(self, features, threshold=0.5):
        # ✅ FIX 3: Call the predict_proba method as a function by passing 'features'
        return (self.predict_proba(features) >= threshold).astype(int)

## 8. Accuracy

**Concept link:** Classification evaluation.

The intended calculation is the mean of the Boolean comparison `labels == predictions`.

**Important notebook issue:** The supplied function does not contain `return`, so the displayed result is `None`. The cell that reports test accuracy therefore prints `None`. This is an actual bug in the supplied notebook, not a change made in these annotations.


**Professor's Code — concept linked to the annotation above**

In [51]:
# how we are defining the accuracy 
def accuracy(labels, predictions):
    np.mean(labels == predictions)

## 9. Confusion counts

**Concept link:** Classification error analysis.

The code counts:

- **TP:** actual 1, predicted 1
- **TN:** actual 0, predicted 0
- **FP:** actual 0, predicted 1
- **FN:** actual 1, predicted 0

**Why:** Accuracy gives one overall number; these counts show what kinds of correct and incorrect predictions occurred.

**Professor's Code — concept linked to the annotation above**

In [52]:
def confusion_counts(labels, predictions):
    true_positive = np.sum((labels == 1) & (predictions == 1))
    true_negative = np.sum((labels == 0) & (predictions == 0))
    false_positive = np.sum((labels == 0) & (predictions == 1))
    false_negative = np.sum((labels == 1) & (predictions == 0))
    return {
        "TP": int(true_positive),
        "TN": int(true_negative),
        "FP": int(false_positive),
        "FN": int(false_negative),
    }

## 10. Generate synthetic student data

**Concept link:** Building a demonstration dataset.

The notebook creates 300 artificial students using a random generator with seed `42`.

Approximate ranges used:

- study hours: 0–20
- attendance: 45–100
- previous score: 30–95

**Why:** The lecture needs a controlled dataset so the internal logistic-regression mechanics can be demonstrated.

**Professor's Code — concept linked to the annotation above**

In [53]:
# generate artificial data
rng = np.random.default_rng(42)
student_count = 300
study_hours = rng.uniform(0,20, student_count)
attendance = rng.uniform(45,100,student_count)
previous_score = rng.uniform(30,95, student_count)

### Inspecting attendance

This print cell displays the generated attendance values. It is an inspection/debugging step rather than a new modelling concept.

In [54]:
print(f"Attendance shape: {attendance.shape}")
print(f"First 10 attendance values: {attendance[:10]}")

Attendance shape: (300,)
First 10 attendance values: [87.8447995  52.40037146 74.48374198 73.28225786 92.16646793 70.45396511
 66.17992229 80.17597992 59.65548246 52.6872626 ]


## 11. Build the feature matrix

**Concept link:** Feature representation.

`np.column_stack(...)` places the three one-dimensional feature arrays into columns. Each row therefore represents one student:

`[study_hours, attendance, previous_score]`

**Why:** The model expects a 2-D feature matrix so that rows are observations and columns are features.

**Professor's Code — concept linked to the annotation above**

In [55]:
# creating features 
features = np.column_stack(
    [study_hours, attendance, previous_score]
)

### Inspecting the feature matrix

This print cell verifies the resulting matrix structure. The important shape idea is:

`number of students × number of features`.

In [56]:
print(f"Feature matrix shape: {features.shape}")
print("First 5 rows:\n", features[:5])

Feature matrix shape: (300, 3)
First 5 rows:
 [[15.47912097 87.8447995  61.03788179]
 [ 8.7775688  52.40037146 79.49311661]
 [17.1719584  74.48374198 88.71631168]
 [13.94736058 73.28225786 76.84523654]
 [ 1.88354696 92.16646793 92.60872953]]


## 12. Create the underlying synthetic score

**Concept link:** Linear score.

The notebook constructs an artificial score using a weighted sum of the three features plus an intercept:

$$
\text{true\_score}=-8.0+0.22x_1+0.055x_2+0.045x_3
$$

**What:** Produces a continuous score from the synthetic student features.

**Why:** This score is then passed through the sigmoid to create a probability for the synthetic labels.

**Important:** This is the notebook's artificial data-generation rule; it is not a learned model parameter from the later training step.

**Professor's Code — concept linked to the annotation above**

In [57]:
# execution 
true_score = (
    -8.0 + 0.22 * study_hours + 0.055 * attendance + 0.045 * previous_score
)

## 13. Convert the synthetic score into a probability

**Concept link:** Sigmoid probability.

`sigmoid(true_score)` converts the synthetic linear score into a value between 0 and 1. That value becomes the probability used to generate the binary labels.

**Professor's Code — concept linked to the annotation above**

In [58]:
pass_probability = sigmoid(true_score)

## 14. Generate labels and create the train/test split

**Concept link:** Dataset splitting.

The labels are sampled from a Bernoulli/binomial process using `pass_probability`. Then the row indices are randomly permuted and split into 80% training and 20% testing.

**Why:** Training and testing must use separate observations so the test set can evaluate the trained model on unseen examples.

**Common beginner mistake:** shuffling features and labels independently, which would break the student-to-label correspondence.

**Professor's Code — concept linked to the annotation above**

In [59]:
labels = rng.binomial(1,pass_probability)
indices = rng.permutation(student_count)
split_index = int(0.8 * student_count)
train_indices = indices[:split_index]
test_indices = indices[split_index:]

## 15. Create training and test datasets

The same index sets are applied to both `features` and `labels`, preserving alignment.

`features_train` ↔ `labels_train` and `features_test` ↔ `labels_test`.

**Professor's Code — concept linked to the annotation above**

In [60]:
features_train = features[train_indices]
features_test = features[test_indices]
labels_train = labels[train_indices]
labels_test = labels[test_indices]

## 16. Standardize features

**Concept link:** Feature scaling.

The notebook calculates the mean and standard deviation from the **training set** and uses those same statistics for both training and test data.

$$
x_{scaled}=\frac{x-\mu_{train}}{\sigma_{train}}
$$

**Why:** The model receives features on a more comparable numerical scale, and the test transformation does not learn new scaling statistics from the test set.

**Common beginner mistake:** computing a separate mean/std from the test set.

**Professor's Code — concept linked to the annotation above**

In [61]:
train_mean = features_train.mean(axis=0)
train_std = features_train.std(axis=0)
features_train_scaled = (features_train - train_mean) / train_std
features_test_scaled = (features_test - train_mean) / train_std

## 17. Train the logistic-regression model

**Concept link:** Gradient-based parameter learning.

The model is created with `learning_rate=0.1` and `epocs=200`. The spelling `epocs` is part of the supplied class API.

`fit(...)` then repeats the forward calculation, loss-gradient calculation, and parameter update for the configured number of epochs.

**Why:** This is where the model learns weights and bias from the scaled training data.

**Professor's Code — concept linked to the annotation above**

In [62]:
# logic regression
model = LogisticRegressionModel(learning_rate=0.1,epocs=200)
model.fit(features_train_scaled,labels_train)

## 18. Generate test probabilities and predictions

**Concept link:** Inference.

- `predict_proba(...)` returns the model's probability output.
- `predict(...)` applies the threshold (0.5 here) to produce binary predictions.

**Key distinction:** probability and class are two different outputs from the pipeline.

**Professor's Code — concept linked to the annotation above**

In [63]:
test_probabilites = model.predict_proba(features_test_scaled)
test_predictions = model.predict(features_test_scaled,threshold=0.5)

## 19. Evaluate the trained model

This cell reports:

- final training loss
- test accuracy
- confusion counts
- learned weights
- learned bias

### Important observation from the supplied output

The confusion counts are:

`TP=39, TN=11, FP=5, FN=5`

The printed test accuracy is `None` because the `accuracy()` function above does not return its computed mean. This should be fixed in the notebook if you want the numerical accuracy to display.

**Professor's Code — concept linked to the annotation above**

In [64]:
# final training loss
print("Final Training Loss : ", round(model.loss_history[-1],4))
# Test accuracy
print("Test Accuracy : ", accuracy(labels_test,test_predictions),4)
# Confusion Counts
print("Confusion Counts : ", confusion_counts(labels_test,test_predictions))
# Weights 
print("Weights : ", np.round(model.weights, 3))
# Bias 
print("Bias : ", round(model.bias, 3))

Final Training Loss :  16.6937
Test Accuracy :  None 4
Confusion Counts :  {'TP': 32, 'TN': 16, 'FP': 7, 'FN': 5}
Weights :  [1.048 0.687 0.573]
Bias :  0.545


## 20. Predict a new student's passing probability

**Concept link:** Prediction on unseen input.

The new student is:

- study hours = 10.0
- attendance = 82.0
- previous score = 67.0

The student is standardized using the **training** mean and standard deviation before prediction.

The supplied output is approximately:

**Passing probability = 0.8096**

This is the model's predicted probability for class 1, not a guaranteed real-world outcome.

**Professor's Code — concept linked to the annotation above**

In [65]:
new_student = np.array([[10.0,82.0,67.0]])
new_student_scaled = (new_student - train_mean) / train_std
new_probability = model.predict_proba(new_student_scaled)[0]
print("new student's passing probability is ", round(new_probability,4))

new student's passing probability is  0.7654


## 21. Convert the new student's probability into a class

The model's default threshold is 0.5. The supplied output is:

`1`

So, under this model and threshold, the new student is classified as class 1 (pass).

The complete inference path is:

`new features → training-statistic scaling → linear score → sigmoid probability → threshold → class`.

**Professor's Code — concept linked to the annotation above**

In [66]:
# model ouput 
print("the student will ", model.predict(new_student_scaled)[0])

the student will  1


## 22. End of supplied notebook

No additional code was supplied in the final cell.

# Quick Code Revision

| Stage | Notebook cell | Core idea |
|---|---:|---|
| Probability | 4 | Sigmoid |
| Loss | 6 | Binary cross-entropy |
| Training | 8 | Gradient descent |
| Evaluation | 9–10 | Accuracy + confusion counts |
| Data | 11–14 | Synthetic features |
| Split | 17–18 | 80/20 train/test |
| Scaling | 19 | Training statistics reused on test data |
| Fit | 20 | Learn weights and bias |
| Predict | 21 | Probability + thresholded class |
| New student | 23–24 | Inference on unseen input |

## Two notebook issues to remember

1. `accuracy()` computes `np.mean(...)` but does not `return` it, so the reported test accuracy is `None`.
2. `binary_cross_entropy()` uses `np.clip(probabilities, 1e-12, 1e-12)`, which collapses both clipping bounds to the same value. The supplied notebook should be reviewed before treating that implementation as final.
